<h1 align='center'> ELCardio Competition Ensembling Method </h1>

## Setup & Imports

In [1]:
pip install iterative-stratification

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string
import re
import nltk
import unicodedata
import itertools
import torch
import torch.nn.functional as F

from tqdm import tqdm
from torch import nn
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import precision_recall_fscore_support, classification_report, f1_score
from tqdm import trange
from nltk import tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import CountVectorizer
from dataclasses import dataclass
from sklearn.preprocessing import MultiLabelBinarizer
from typing import List, Dict, Optional, Iterator, Any

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Data Loading & Parsing

### Dataclasses

In [4]:
@dataclass
class Mention:
    """
    Represents a mention-level annotation within the clinical text.
    
    Attributes:
        start (int): The starting character offset in the document.
        end (int): The ending character offset in the document.
        code (str): The assigned ICD-10 diagnostic code.
        text (str): The actual text span (e.g., "συγκοπτικό επεισόδιο").
    """
    start: int
    end: int
    code: str
    mention: str


@dataclass
class ClinicalDocument:
    """
    Represents a complete clinical document (discharge letter) from the dataset.
    
    Attributes:
        patient_id (Any): Unique identifier for the patient/record.
        text (str): The raw unstructured clinical narrative.
        document_labels (List[List[str]]): A list of lists containing valid ICD-10 codes 
            for the document. The system must predict at least one code from each inner list.
        mentions (Optional[List[Mention]]): A list of mention-level annotations, if available.
    """
    patient_id: Any
    text: str
    document_labels: List[List[str]]
    mentions: Optional[List[Mention]] = None

### Read from jsonl

In [5]:
filepath = "/kaggle/input/datasets/dimitriospakakis/elcardio-dataset/trainELCardioCC2026.jsonl"
all_documents= []
with open(filepath, 'r', encoding='utf-8') as f:
    for line in f:
       json_line = json.loads(line)

       # Load the basic data
       patient_id = json_line['patient_id']
       text = json_line['text']
       document_level_annotations = json_line['document_level_annotations']

       # Load the mentions
       raw_mentions = json_line['mention_level_annotations'] 

       final_mentions = None
      
       # If the mentions list is not empty then we create Mention objects 
       if raw_mentions:
         final_mentions = [Mention(m['start'], m['end'], m['code'], m['mention']) for m in raw_mentions]
       
       all_documents.append(ClinicalDocument(patient_id,text,document_level_annotations,final_mentions))
        
print(len(all_documents))


2500


## Exploratory Data Analysis

In [6]:
def fix_mention_offsets(document_text: str, mention: Mention) -> Mention:
    """
    Fix the wrong offsets looking for the rigth string in a window of +/- characters
    """
    # the correct string
    target_text = mention.mention
    
    # Define a search window
    search_start = max(0, mention.start - 15)
    search_end = min(len(document_text), mention.end + 15)
    search_window = document_text[search_start:search_end]
    
    # look up fot the string 
    local_start = search_window.find(target_text)
    
    if local_start != -1:
        # calculate the correct offsets
        new_start = search_start + local_start
        new_end = new_start + len(target_text)
        
        
        return Mention(start=new_start, end=new_end, code=mention.code, mention=target_text)
    else:
        return mention

In [7]:
for document in all_documents:
    doc_mentions = document.mentions
    new_mentions = []
    if doc_mentions:
        for ment in doc_mentions:
          if not ment.end - ment.start < 2:
            new_mentions.append(fix_mention_offsets(document.text,ment))
    document.mentions=new_mentions


all_mentions = []
for doc in all_documents:
    if doc.mentions:
        all_mentions.append(doc.mentions)

print(f"{len(all_mentions)} mention-level annotations found")

1493 mention-level annotations found


In [8]:
count= 0
for document in all_documents:
    ment = document.mentions
    if ment:
        for m in ment:
            new_m = fix_mention_offsets(document.text,m)
            st = new_m.start
            en = new_m.end
            txt = new_m.mention
            sliced_text = document.text[st:en]
            if sliced_text != txt:
                count+=1
print(f"{count} wrong offsets")

0 wrong offsets


#### Remove overlaps

In [9]:
overlaps_removed = 0

for doc in all_documents:
    if not doc.mentions:
        continue
        
    # Sort the mentions in a way that the longest mention is at the 1st position
    sorted_mentions = sorted(doc.mentions, key=lambda m: (m.start, -m.end))
    
    clean_mentions = []
    
    for m in sorted_mentions:
        if not clean_mentions:
            clean_mentions.append(m)
        else:
            last_mention = clean_mentions[-1]
            
            if m.start < last_mention.end:
                overlaps_removed += 1
                continue
            else:
                clean_mentions.append(m)
                
    doc.mentions = clean_mentions

print(f"{overlaps_removed} redundant/overlapping mentions were deleted.")

3 redundant/overlapping mentions were deleted.


## Dataset Split

We will split the dataset to 90% train and 5% validation and 5% test. We will handle the codes that are rare. If a code has only 1 document, it will be in the training set. If there are more than 2 documents then there will be an uniform distribution in both test, train and validation sets

We can not convert the y_labels list to numpy array or tensor because every document has a different number of codes, but the 2nd dimension must be the same for all the samples. So, to improve the performance of the training and to solve this issue, we will convert the codes with multi-Hot Encoding. Each label list will be converted to a list where the true codes will be converted to 1 and the false to 0.

In [10]:
X_texts = []
y_labels_raw = []
y_labels = []
for doc in all_documents:
    X_texts.append(doc.text)
    
    current_labels = []
    for label_list in doc.document_labels:
        for label in label_list:
            current_labels.append(label)
    y_labels.append(current_labels.copy())
    # Add extra flag to stratify the set using the labels and the mentions
    if doc.mentions and len(doc.mentions) > 0:
        current_labels.append("HAS_MENTION")
        
    y_labels_raw.append(current_labels)

X_texts = np.array(X_texts)

# Binarization
mlb = MultiLabelBinarizer()
y_encoded = mlb.fit_transform(y_labels_raw)

mlb2 = MultiLabelBinarizer()
y_labels = mlb2.fit_transform(y_labels)

# 3. Stratified Split (90% Train, 5% Val, 5% Test)
X_indices = np.arange(len(all_documents))

# --- Split: 90% Train / 10% Temp ---
msss_1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_idx, temp_idx = next(msss_1.split(X_indices, y_encoded))

# --- Split: 5% Val / 5% Test ---
msss_2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_local_idx, test_local_idx = next(msss_2.split(temp_idx, y_encoded[temp_idx]))

val_idx = temp_idx[val_local_idx]
test_idx = temp_idx[test_local_idx]

train_docs = [all_documents[i] for i in train_idx]
val_docs = [all_documents[i] for i in val_idx]
test_docs = [all_documents[i] for i in test_idx]

y_train = y_labels[train_idx]
y_val = y_labels[val_idx]
y_test = y_labels[test_idx]

string_labels = mlb2.classes_
# Εκτυπώσεις
print(f"y_labels shape: {y_labels.shape}")
print(f"Example y_labels[0]: {y_labels[0]}")
print(f"Labels document 0: {string_labels[y_labels[0] == 1]}\n")

print(f"Total documents: {len(all_documents)}")
print("-" * 30)
print(f"Train Set:      {len(train_docs)}")
print(f"Validation Set: {len(val_docs)}")
print(f"Test Set:       {len(test_docs)}")

y_labels shape: (2500, 110)
Example y_labels[0]: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0]
Labels document 0: ['I10' 'I11' 'I44' 'R55' 'Z95']

Total documents: 2500
------------------------------
Train Set:      2238
Validation Set: 134
Test Set:       128


In [11]:
tokenizer = AutoTokenizer.from_pretrained('nlpaueb/bert-base-greek-uncased-v1', do_lower_case = True, strip_accents = True)

config.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## Tokenization/ Sliding Window/ BIO Tagging

We will use the greekBERT tokenizer to tokenize the text. In the previous section we saw that a lot of documents have more than 512 tokens. So we will use a sliding window technique to convert the big documents to chunks. After we will tag the tokens with the mentions we have using the BIO tagging technique. The fact that we will create chunks from the documents creates a problem with the labels. There is not a single way to solve this , so we will experiment with this

In [12]:
def prepare_features(docs, y_labels, tokenizer, string_labels):
    all_input_ids = []
    all_attention_masks = []
    all_ner_labels = []     #  BIO tags
    all_class_labels = []   # multi-hot encoding
    all_patient_ids = []

    for doc_idx, doc in enumerate(docs):
        doc_y = y_labels[doc_idx]
        
        # 2. Tokenization & Sliding Window
        encoded = tokenizer(
            doc.text,
            padding='max_length',
            truncation=True,
            max_length=512,
            stride=64,
            return_overflowing_tokens=True,
            return_offsets_mapping=True
        )
        
        # for every chunk in the document
        for chunk_idx in range(len(encoded['input_ids'])):
            chunk_input_ids = encoded['input_ids'][chunk_idx]
            chunk_mask = encoded['attention_mask'][chunk_idx]
            chunk_offsets = encoded['offset_mapping'][chunk_idx]
            
            
            # Initialize BIO tags
            chunk_ner_tags = ['O'] * 512
            
            # Initialize the labels 
            chunk_class_label = np.zeros(110, dtype=int)
            
            # If the document has mention annotation for the labels
            if doc.mentions and len(doc.mentions) > 0:
                for i, (start, end) in enumerate(chunk_offsets):
                    
                    # Check if special token
                    if start==end==0:
                        chunk_ner_tags[i] = -100
                        continue
                    
                    for mention in doc.mentions:
                        if start >= mention.start and end <= mention.end:
                            if i == 0 or chunk_ner_tags[i-1] not in [f"B-{mention.code}", f"I-{mention.code}"]:
                                chunk_ner_tags[i] = f"B-{mention.code}"
                            else:
                                chunk_ner_tags[i] = f"I-{mention.code}"
                            index = np.where(string_labels == mention.code)
                            chunk_class_label[index] = 1
                            break
                    pass 
            else:
                # If the document does not have mentions then we set all tags to -100 so pytorch will ignore them
                chunk_ner_tags = [-100] * 512
                
                chunk_class_label = doc_y
                pass
                
            # Save the chunks results to the lists
            all_patient_ids.append(doc.patient_id)
            all_input_ids.append(chunk_input_ids)
            all_attention_masks.append(chunk_mask)
            all_ner_labels.append(chunk_ner_tags)
            all_class_labels.append(chunk_class_label)

    return all_input_ids, all_attention_masks, all_ner_labels, all_class_labels, all_patient_ids
    

## Convert the data to tensors

Let's say that this section is where analysis and experiments stop and starts the 'real' nlp task. To summarize our previous work:

1. We have 3 sets. Training/Validation/Test.

2. We have 4 lists: input_ids, attention_masks, ner_labels, class_labels, for each set.

3. We used a sliding window to create tokens_sets of max 512 tokens and to keep all information, we didnt truncate but we splitted the documents to chunks

4. The chunks of documents with mentions will have the labels from the mentions for each chunk

5. The chunks of documents without mentions will inherit the labels from the global document (it is possible that this way add false information).

#### Convert BIO Tags to numbers

Create tag to id dictionary

In [13]:
tag2id_dict = {}
tag2id_dict['O'] = 0
counter = 0
for code in string_labels:
    counter+=1
    tag2id_dict['B-' + code] = counter
    counter+=1
    tag2id_dict['I-' + code] = counter


In [14]:
def convert_tag2id(ner_labels,tag2id_dict):
    id_list = []
    for i in range(len(ner_labels)):
        if ner_labels[i] != -100:
            id_list.append(tag2id_dict[ner_labels[i]])
        else:
            id_list.append(-100)
    return id_list

#### Create tensors

In [15]:
def create_tensors(x_docs,y,tokenizer,string_labels,tag2id_dict):
    input_ids, attention_masks, ner_labels, labels , patients_ids= prepare_features(x_docs,y,tokenizer,string_labels)
    for i in range(len(ner_labels)):
        ner_labels[i] = convert_tag2id(ner_labels[i],tag2id_dict)

    input_ids = np.array(input_ids)
    attention_masks = np.array(attention_masks)
    ner_labels = np.array(ner_labels)
    labels = np.array(labels)
    patients_ids = np.array(patients_ids)
    return torch.tensor(input_ids).long() , torch.tensor(attention_masks).long() , torch.tensor(ner_labels).long() , torch.tensor(labels, dtype=torch.float32) , torch.tensor(patients_ids, dtype=torch.int)

In [16]:
train_input_ids_tensor, train_attention_masks_tensor, train_ner_labels_tensor, y_train_tensor, patient_ids_train = create_tensors(train_docs,y_train,tokenizer,string_labels,tag2id_dict)
val_input_ids_tensor, val_attention_masks_tensor, val_ner_labels_tensor, y_val_tensor, patient_ids_val = create_tensors(val_docs,y_val,tokenizer,string_labels,tag2id_dict)
test_input_ids_tensor, test_attention_masks_tensor, test_ner_labels_tensor, y_test_tensor , patient_ids_test= create_tensors(test_docs,y_test,tokenizer,string_labels,tag2id_dict)

#### Create TensorDataset and DataLoader

In [17]:
BATCH_SIZE = 16

train_dataset = TensorDataset(train_input_ids_tensor, train_attention_masks_tensor, train_ner_labels_tensor, y_train_tensor, patient_ids_train)
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
val_dataset = TensorDataset(val_input_ids_tensor, val_attention_masks_tensor, val_ner_labels_tensor, y_val_tensor,patient_ids_val)
val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False)
test_dataset = TensorDataset(test_input_ids_tensor, test_attention_masks_tensor, test_ner_labels_tensor, y_test_tensor, patient_ids_test)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False)


## Model

We will use the GreekBERT from hugging face

In [18]:
class ElCardioModel(nn.Module):
    def __init__(self):
        super(ElCardioModel, self).__init__()

        # Greek Bert with no heads
        self.bert = AutoModel.from_pretrained("nlpaueb/bert-base-greek-uncased-v1")

        # Dropout for regularization
        self.dropout = nn.Dropout(p=0.1)

        # Classification Head
        self.classification_head = nn.Linear(768,110)

        # NER Head
        self.ner_head = nn.Linear(768,221)


    def forward(self, input_ids, attention_mask):
        
        bert_output = self.bert(input_ids,attention_mask)
        sequence_bert_output = bert_output.last_hidden_state 

        sequence_bert_output = self.dropout(sequence_bert_output)
        # classification head
        classification = self.classification_head(sequence_bert_output[:, 0, :])

        # NER head
        ner = self.ner_head(sequence_bert_output)

        return classification, ner

## Testing

In [19]:
def get_document_probabilities(model_instance, dataloader, has_targets=True):
    """
    From a dataloader and a model, extract probabilities 
    and Document-Level Max Pooling.
    """
    model_instance.eval()
    probs_list, targets_list, patient_ids_list = [], [], []
    
    with torch.no_grad():
        for batch in dataloader:
            b_input_ids = batch[0].to(device)
            b_attn_mask = batch[1].to(device)
            
            # If final test set, then we dont have targets
            if has_targets:
                b_class_labels = batch[3].to(device)
                b_patient_ids = batch[4]
            else:
                b_patient_ids = batch[2]
            
            logits_class, _ = model_instance(b_input_ids, b_attn_mask)
            probs = torch.sigmoid(logits_class)
            
            probs_list.append(probs.cpu().numpy())
            patient_ids_list.append(b_patient_ids.numpy())
            if has_targets:
                targets_list.append(b_class_labels.cpu().numpy())
                
    # Max Pooling
    df_probs = pd.DataFrame(np.vstack(probs_list))
    df_probs['patient_id'] = np.concatenate(patient_ids_list)
    doc_probs = df_probs.groupby('patient_id').max().values
    
    unique_ids = df_probs.groupby('patient_id').max().index.values
    
    if has_targets:
        df_targets = pd.DataFrame(np.vstack(targets_list))
        df_targets['patient_id'] = np.concatenate(patient_ids_list)
        doc_targets = df_targets.groupby('patient_id').max().values
        return doc_probs, doc_targets, unique_ids
    else:
        return doc_probs, None, unique_ids

### Load Models

In [20]:
model_micro = ElCardioModel().to(device)
model_micro.load_state_dict(torch.load('/kaggle/input/notebooks/dimitriospakakis/elcardio-maxmicro/best_elcardio_model_final.pth'))

model_macro = ElCardioModel().to(device)
model_macro.load_state_dict(torch.load('/kaggle/input/notebooks/dimitriospakakis/elcardio-maxmacro-basic/best_elcardio_model_maxmacro_basic.pth'))

pytorch_model.bin:   0%|          | 0.00/454M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/454M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpaueb/bert-base-greek-uncased-v1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpaueb/bert-base-greek-uncased-v1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<All keys matched successfully>

## Threhsold Tuning

### Extract Probabilities

In [21]:
val_probs_micro, val_targets, _ = get_document_probabilities(model_micro, val_loader)
val_probs_macro, _, _ = get_document_probabilities(model_macro, val_loader)

### Ensemble

In [39]:
W_MICRO = 0.4
W_MACRO = 0.6
val_probs_ens = (W_MICRO * val_probs_micro) + (W_MACRO * val_probs_macro)

### Global Tuning (micro f1)

In [40]:
best_global_f1 = -1
best_global_th = 0.5 

for th in np.arange(0.30, 0.72, 0.01):
    preds = (val_probs_ens > th).astype(int)
    f1 = f1_score(val_targets, preds, average='micro', zero_division=0)
    if f1 > best_global_f1:
        best_global_f1 = f1
        best_global_th = th

print(f"Optimal Global Threshold: {np.round(best_global_th, 2)} (Micro F1: {best_global_f1:.4f})")

Optimal Global Threshold: 0.52 (Micro F1: 0.7810)


### Per class Tuning (macro f1)

In [41]:
num_classes = val_targets.shape[1] 
best_per_class_thresholds = np.zeros(num_classes)

for i in range(num_classes):
    best_f1_cls = -1
    best_th_cls = 0.5 
    
    if np.sum(val_targets[:, i]) == 0:
        best_per_class_thresholds[i] = 0.5
        continue
        
    for th in np.arange(0.05, 0.96, 0.01):
        preds = (val_probs_ens[:, i] > th).astype(int)
        f1 = f1_score(val_targets[:, i], preds, zero_division=0)
        
        if f1 > best_f1_cls:
            best_f1_cls = f1
            best_th_cls = th
            
    best_per_class_thresholds[i] = best_th_cls

print("Optimal Per-Class Thresholds (Top 10):", np.round(best_per_class_thresholds[:10], 2))

Optimal Per-Class Thresholds (Top 10): [0.44 0.5  0.56 0.5  0.57 0.5  0.64 0.46 0.08 0.58]


## Evaluation on the test set

In [42]:
print("\nExtract Probabilities from Internal Test Set...")
test_probs_micro, test_targets, _ = get_document_probabilities(model_micro, test_loader)
test_probs_macro, _, _ = get_document_probabilities(model_macro, test_loader)

test_probs_ens = (W_MICRO * test_probs_micro) + (W_MACRO * test_probs_macro)

print("\n" + "="*50)
print("EVALUATION ON INTERNAL TEST SET (ENSEMBLE)")
print("="*50)

# Default 0.5
preds_default = (test_probs_ens > 0.5).astype(int)
mic_p, mic_r, mic_f1, _ = precision_recall_fscore_support(test_targets, preds_default, average='micro', zero_division=0)
mac_p, mac_r, mac_f1, _ = precision_recall_fscore_support(test_targets, preds_default, average='macro', zero_division=0)
print(f"[Default 0.50] Micro F1: {mic_f1:.4f} | Macro F1: {mac_f1:.4f}")

# Global Threshold
preds_global = (test_probs_ens > best_global_th).astype(int)
mic_p, mic_r, mic_f1, _ = precision_recall_fscore_support(test_targets, preds_global, average='micro', zero_division=0)
mac_p, mac_r, mac_f1, _ = precision_recall_fscore_support(test_targets, preds_global, average='macro', zero_division=0)
print(f"[Global Th={best_global_th:.2f}] Micro F1: {mic_f1:.4f} | Macro F1: {mac_f1:.4f}")

# Per-Class Thresholds
preds_perclass = (test_probs_ens > best_per_class_thresholds).astype(int)
mic_p, mic_r, mic_f1, _ = precision_recall_fscore_support(test_targets, preds_perclass, average='micro', zero_division=0)
mac_p, mac_r, mac_f1, _ = precision_recall_fscore_support(test_targets, preds_perclass, average='macro', zero_division=0)
print(f"[Per-Class Th] Micro F1: {mic_f1:.4f} | Macro F1: {mac_f1:.4f}")


Extract Probabilities from Internal Test Set...

EVALUATION ON INTERNAL TEST SET (ENSEMBLE)
[Default 0.50] Micro F1: 0.8111 | Macro F1: 0.3818
[Global Th=0.52] Micro F1: 0.8121 | Macro F1: 0.3771
[Per-Class Th] Micro F1: 0.7657 | Macro F1: 0.3593


## Final Answer

In [43]:
print("\nLoading Final Test Set...")
test_file_path = "/kaggle/input/datasets/dimitriospakakis/final-test-set/test_set.jsonl" 

test_documents = []
with open(test_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        test_documents.append(ClinicalDocument(patient_id=data['patient_id'], text=data['text'], document_labels=[], mentions=[]))

dummy_y = np.zeros((len(test_documents), len(string_labels)))

ext_test_input_ids, ext_test_attention_masks, _, _, ext_test_patients_ids = create_tensors(
    x_docs=test_documents, y=dummy_y, tokenizer=tokenizer, string_labels=string_labels, tag2id_dict=tag2id_dict
)

ext_test_dataset = TensorDataset(ext_test_input_ids, ext_test_attention_masks, ext_test_patients_ids)
ext_test_loader = DataLoader(ext_test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Inference in Test Set...")
ext_probs_micro, _, unique_patient_ids = get_document_probabilities(model_micro, ext_test_loader, has_targets=False)
ext_probs_macro, _, _ = get_document_probabilities(model_macro, ext_test_loader, has_targets=False)

ext_probs_ens = (W_MICRO * ext_probs_micro) + (W_MACRO * ext_probs_macro)

# 1.  Default: final_predictions = (ext_probs_ens > 0.5).astype(int)
# 2.  Global:  final_predictions = (ext_probs_ens > best_global_th).astype(int)
# 3.  Per-Class: final_predictions = (ext_probs_ens > best_per_class_thresholds).astype(int)

# Επιλέγουμε το default 0.5 καθώς συνήθως γενικεύει καλύτερα στα Ensemble
final_predictions = (ext_probs_ens > 0.5).astype(int) 
# ------------------------------------------------------------------------

submission_records = []
for i, patient_id in enumerate(unique_patient_ids):
    patient_preds = final_predictions[i]
    predicted_indices = np.where(patient_preds == 1)[0]
    predicted_codes = [string_labels[idx] for idx in predicted_indices]
    
    record = {
        "patient_id": int(patient_id),
        "annotations": predicted_codes
    }
    submission_records.append(record)

submission_filename = 'submission_ensemble_weighted.jsonl'
with open(submission_filename, 'w', encoding='utf-8') as f:
    for record in submission_records:
        f.write(json.dumps(record) + '\n')
print(f"The submision file {submission_filename} is ready!")


Loading Final Test Set...
Inference in Test Set...
The submision file submission_ensemble_weighted.jsonl is ready!
